# Assignment 3: Assemble IR and LM for RAG

## Overview
Assemble the BM25 retriever from Assignment 1 and the text generator from Assignment 2 to build a RAG system. Test it with open-form questions on the Cranfield collection. This assignment covers four tasks: building the RAG system, selecting the best generator, understanding how retrieval affects output quality, and diagnosing where the pipeline succeeds or fails. All evaluations in this assignment are qualitative. In your analysis, do not just describe what the model outputs, but engage with why it behaves the way it does and what it reveals about the system.

## Requirements
- Make sure all cells have been run and outputs are visible. Queries and model responses **must be displayed** in the notebook output.
- Implement each task in the cells marked **TODO** and answer questions marked **TODO**

## Deadline
**May 1, 23:59 CET**

## Submission
- Submit only the .ipynb file. Add your name to the filename.

In [1]:
# Load imports
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import torch
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForCausalLM
import ir_datasets


/home/spark-dasha/miniconda3/envs/rags/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the models and tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_full   = AutoModelForCausalLM.from_pretrained("/home/spark-dasha/Documents/rag/Assignment 2/ckpt-full")

model_completion = AutoModelForCausalLM.from_pretrained("/home/spark-dasha/Documents/rag/Assignment 2/ckpt-completion")

print("Both models loaded.")

Both models loaded.


## Task 1: Build a RAG System

### Task overview
1. Implement a `BM25` class (ported from Assignment 1) and build an index over the Cranfield collection.
2. Implement a `RAGSystem` class that ties retrieval and generation together.




### Task 1.1: BM25 Retriever

Port your `BM25` implementation from Assignment 1, keeping only the attributes and methods you require.

In [3]:
class BM25:
    """
    Minimal BM25 retriever (ported from Assignment 1).

    Brute-force scoring over all documents (no inverted index).
    Only the attributes/methods required by the RAG pipeline are kept:
      - tokenize(text)
      - build(docs_iter)
      - idf(term), score(query_tokens, doc_id)
      - retrieve(query_text, k)
    The ``docs_store`` mapping is kept so that the RAG system can look up
    the raw title/text of each retrieved document.
    """

    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        # BM25 hyperparameters
        self.k1 = k1                # term-frequency saturation
        self.b = b                  # document length normalisation
        self.remove_stopwords = remove_stopwords

        # Tokenization helpers
        self._stemmer = PorterStemmer()
        self._stopwords = set(stopwords.words("english")) if remove_stopwords else set()

        # Corpus statistics (populated by build())
        self.N: int = 0
        self.avgdl: float = 0.0
        self.df: Dict[str, int] = {}
        self.doc_len: Dict[str, int] = {}
        self.doc_tf: Dict[str, Counter] = {}
        self.docs_store: Dict[str, Dict[str, str]] = {}

    # -------------------------- tokenization --------------------------
    def tokenize(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.findall(r"\b\w+\b", text)
        out = []
        for tok in tokens:
            if tok in self._stopwords:
                continue
            stem = self._stemmer.stem(tok)
            if stem:
                out.append(stem)
        return out

    # -------------------------- index build ---------------------------
    def build(self, docs_iter: Iterable) -> None:
        for doc in docs_iter:
            doc_id = doc.doc_id
            title = doc.title or ""
            text = doc.text or ""
            tokens = self.tokenize(f"{title} {text}")
            tf = Counter(tokens)
            self.doc_tf[doc_id] = tf
            self.doc_len[doc_id] = sum(tf.values())
            self.docs_store[doc_id] = {"title": title, "text": text}

        df_counter = Counter()
        for tf in self.doc_tf.values():
            df_counter.update(set(tf.keys()))
        self.df = dict(df_counter)
        self.N = len(self.doc_tf)
        self.avgdl = float(np.mean(list(self.doc_len.values()))) if self.N > 0 else 0.0

    # -------------------------- scoring -------------------------------
    def idf(self, term: str) -> float:
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query_tokens: List[str], doc_id: str) -> float:
        doc_tf = self.doc_tf.get(doc_id, Counter())
        dl = self.doc_len.get(doc_id, 0)
        if dl == 0:
            return 0.0
        s = 0.0
        for term in query_tokens:
            f = doc_tf.get(term, 0)
            if f == 0:
                continue
            num = f * (self.k1 + 1)
            den = f + self.k1 * (1 - self.b + self.b * (dl / self.avgdl))
            s += self.idf(term) * (num / den)
        return s

    # -------------------------- retrieval -----------------------------
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
        q_tokens = self.tokenize(query_text)
        scores = []
        for doc_id in self.doc_tf.keys():
            s = self.score(q_tokens, doc_id)
            if s > 0:
                scores.append((doc_id, s))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]

### Build the Cranfield Index

Load the Cranfield corpus with `ir_datasets` and call `bm25.build()`. You may use the best `k1` and `b` hyperparameters from Assignment 1.


In [5]:
# Load the Cranfield corpus and build the BM25 index
dataset = ir_datasets.load("cranfield")

# Best hyperparameters from Assignment 1 grid-search.
bm25 = BM25(k1=1.80, b=0.50)
bm25.build(dataset.docs_iter())

print(f"Index built: {bm25.N} documents, avg doc length = {bm25.avgdl:.1f} tokens")

[INFO] [starting] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[INFO] [finished] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz: [00:00] [507kB] [2.36MB/s]


Index built: 1400 documents, avg doc length = 102.9 tokens


### Task 1.2: RAGSystem

Implement `RAGSystem` with the following behaviour:

* **`build_prompt(query, retrieved_docs)`** — assemble the prompt using the **same instruction-tuning template** as in Assignment 2 (`format_prompt`). Concatenate the retrieved documents as a single doc to accomodate *k* larger than 1.
* **`generate(query, k, verbose)`** — full RAG pipeline:
  1. Retrieve top-*k* docs via `self.bm25.retrieve()`
  2. Build the prompt with `build_prompt`
  3. Run inference; **strip the prompt prefix** from the decoded output
  4. Return the answer string only
  5. Have a *verbose* flag to return the retrieved docs too. It will be useful for evaluation.
The constructor must accept `model`, `tokenizer`, `bm25`, `k` (default 3), and `max_new_tokens` (default 256).


In [ ]:
class RAGSystem:
    """
    Retrieval-Augmented Generation (RAG) pipeline combining BM25 retrieval
    with a fine-tuned causal language model for open-form question answering.

    The pipeline operates in two stages:
      1. Retrieval: the BM25 index is queried to fetch the top-k most
         relevant documents for a given question.
      2. Generation: the retrieved documents are injected into an
         instruction-tuning prompt and passed to the language model, which
         produces a grounded answer.

    Typical usage:
        rag = RAGSystem(model=model, tokenizer=tokenizer, bm25=bm25)
        answer = rag.generate("What causes boundary layer separation?", k=3)
    """

    def __init__(
        self,
        model,
        tokenizer,
        bm25: BM25,
        max_new_tokens: int = 500,
    ):
        """
        Initialise the RAG system.

        Args:
            model          : fine-tuned causal language model used for generation.
            tokenizer      : tokenizer corresponding to *model*
            bm25           : a fully built BM25 instance (``bm25.build()`` must
                             have been called before passing it here).
            max_new_tokens : maximum number of new tokens the model may generate
                             per call to :meth:`generate`.
        """
        self.model = model
        self.tokenizer = tokenizer
        self.bm25 = bm25
        self.max_new_tokens = max_new_tokens
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def build_prompt(
        self,
        query: str,
        retrieved_docs: list,
        instruction: str = None,
    ) -> str:
        """
        Assemble the instruction-tuning prompt used for RAG generation.

        Follows the same three-block template as Assignment 2's ``format_prompt``:
        ``### Instruction``, ``### Input``, ``### Response``.  
        
        Retrieved documents are concatenated (title + body) and inserted as the
        ``Document`` field of the input block, allowing the model to draw on
        multiple passages in a single forward pass.

        **Tip**: If you are unsure what the prompt should look like, inspect a
        few samples from your Assignment 2 test set — the structure used there
        is exactly what this method should replicate (minus the retrieved
        context, which replaces the original document field).

        Args:
            query         : the user question or instruction to be answered.
            retrieved_docs: list of document dicts, each containing ``"title"``
                            and ``"text"`` keys (as stored in ``BM25.docs_store``).
            instruction   : optional override for the ``### Instruction`` block.
                            Defaults to a generic technical-QA instruction when
                            *None*. Look at Test data samples for sample instructions

        Returns:
            Formatted prompt string ready for tokenisation, ending with
            ``"### Response:\\n"`` so the model continues from that position.
        """
        if instruction is None:
            instruction = "Answer the following technical question using only the information in the document."

        # Concatenate retrieved passages (title + body) into a single Document
        # block, mirroring the "Question:\n...\n\nDocument:\n..." input layout
        # used during instruction-tuning in Assignment 2.
        doc_chunks = []
        for d in retrieved_docs:
            title = (d.get("title") or "").strip()
            text = (d.get("text") or "").strip()
            if title:
                doc_chunks.append(f"{title}\n{text}")
            else:
                doc_chunks.append(text)
        document_block = "\n\n".join(doc_chunks)

        input_text = f"Question:\n{query.strip()}\n\nDocument:\n{document_block}"
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )
        return prompt

    def generate(
        self,
        query: str,
        k: int = 3,
        verbose: bool = False,
    ) -> str | tuple:
        """
        Run the full RAG pipeline for a single query.

        Steps:
          1. Retrieve the top-k documents from the BM25 index.
          2. Build the instruction-tuning prompt via `build_prompt`.
          3. Tokenise the prompt and run generation with the language model.
          4. Decode only the newly generated tokens (the prompt prefix is stripped).
          5. Return the answer, optionally alongside the retrieved documents.

        Args:
            query   : raw (un-tokenised) user question.
            k       : number of documents to retrieve and include in the prompt.
            verbose : when *True*, returns a ``(answer, retrieved_docs)`` tuple
                      so callers can inspect which passages informed the answer.

        Returns:
            The generated answer string when verbose is False, or a
            ``(answer, retrieved_docs)`` tuple when verbose is True.
        """
        # 1. Retrieve top-k docs and gather their raw title/text records.
        hits = self.bm25.retrieve(query, k=k)
        retrieved_docs = [self.bm25.docs_store[doc_id] for doc_id, _ in hits]

        # 2. Build the instruction-tuning prompt.
        prompt = self.build_prompt(query, retrieved_docs)

        # 3. Tokenise and generate.
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False,
            )

        # 4. Decode only the newly generated tokens (drop the prompt prefix).
        answer = self.tokenizer.decode(
            output_ids[0][prompt_len:], skip_special_tokens=True
        ).strip()

        # 5. Truncate at the first prompt-format marker to stop repetition loops.
        answer = re.split(r'\n###', answer)[0].strip()

        if verbose:
            return answer, retrieved_docs
        return answer


### Task 1.3: Check your implementation

Run the cell below to verify that the full RAG pipeline works end-to-end.  
**Note:** this cell requires a loaded model and tokenizer — run it after you have loaded at least one checkpoint in Problem 2.

In [7]:
# Full RAG test
# Replace model / tokenizer with whichever checkpoint you loaded first.
SAMPLE_QUERY = "what are the structural and aeroelastic problems associated with flight of high speed aircraft"

rag_test = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

answer, docs = rag_test.generate(SAMPLE_QUERY, k=1, verbose=True)

print("=== Retrieved documents ===")
for i, doc in enumerate(docs, 1):
    title = doc.get("title", "").strip() or "(no title)"
    snippet = doc.get("text", "").replace("\n", " ")
    print(f"[{i}] {title}\n    {snippet}...\n")

print("=== Generated answer ===")
print(answer)

=== Retrieved documents ===
[1] some structural and aerelastic considerations of high
speed flight .
    some structural and aerelastic considerations of high speed flight .   the dominating factors in structural design of high-speed aircraft are thermal and aeroelastic in origin .  the subject matter is concerned largely with a discussion of these factors and their interrelation with one another .  a summary is presented of some of the analytical and experimental tools available to aeronautical engineers to meet the demands of high-speed flight upon aircraft structures .  the state of the art with respect to heat transfer from the boundary layer into the structure, modes of failure under combined load as well as thermal inputs and acrothermoelasticity is discussed .  methods of attacking and alleviating structural and aeroelastic problems of high-speed flight are summarized .  finally, some avenues of fundamental research are suggested ....

=== Generated answer ===
The structural and

---
# Tasks 2–4: Evaluation & Analysis

The remainder of this assignment is about evaluating and analysing the outputs of your RAG system by systematically varying inputs — such as queries, prompt instructions, and retrieval depth — and observing how the outputs change.

## Query Selection

<p style="font-size:1.2em">Manually select queries from the <strong>held-out test set</strong> of your Assignment 2 instruction-tuning data. Do <strong>not</strong> randomly sample from the full Cranfield dataset. If you modify or expand a query (e.g. for better retrieval), you may do so but must describe exactly how you did it. A single query may be reused across <strong>at most two</strong> tasks.</p>

Queries must come from the test set to avoid **data leakage** — if a query was seen during fine-tuning, the model may reproduce a memorised answer rather than reasoning from the retrieved context, making your evaluation unreliable.

---

## Task 2: Generator Evaluation

Compare your two fine-tuned models from Assignment 2 to choose the best generator for your system:
- **RAG A** — fine-tuned with loss over the **full prompt** (instruction + input + response)
- **RAG B** — fine-tuned with loss over the **completion only** (response tokens only)

**Note**: This task evaluates the generator in isolation. Both models receive the same retrieved context. Evaluate the generated responses only, not the quality of the retrieval.

### Implement RAG systems with both the models

In [8]:
# Instantiate one RAGSystem per model using the same BM25 index
rag_A = RAGSystem(model=model_full, tokenizer=tokenizer, bm25=bm25)
rag_B = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

### Part 1: Select Queries

Choose queries covering **all three** conditions below. Fill in your final chosen queries in the code cell (TODO).

| Condition | Minimum | Your queries |
|-----------|---------|---------------|
| Simple factual query | 2 | |
| Query requiring a structured/formatted answer (e.g. bullet points) | 2 | |
| One query tested with two different reformulations (same keywords, different wording / instruction template) | 1 pair | |



In [9]:
# Task 2 — generator-only evaluation.
# Same retrieval depth (k) is used for both models so any differences in
# the outputs come from the generator alone.
K_TASK2 = 3

# Queries are taken from the held-out test set of the instruction-tuning
# dataset (test.jsonl in Assignment 2) so we know they were never seen
# during fine-tuning. Each entry is (label, instruction, query).
task2_queries = [
    # ---- Factual queries (>=2) ----
    (
        "factual-1 (test qid=218)",
        "Answer the following technical question using only the information in the document.",
        "what is the heat transfer to a blunt body in the absence of vorticity .",
    ),
    (
        "factual-2 (test qid=39)",
        "Answer the following technical question using only the information in the document.",
        "how can one detect transition phenomena in boundary layers .",
    ),
    # ---- Structured / formatted answer (>=2) ----
    (
        "structured-1 (test qid=50)",
        "Using only the document below, answer the question in bullet points.",
        "what are the effects of small amounts of gas rarefaction on the characteristics of the boundary layers on slender bodies .",
    ),
    (
        "structured-2 (test qid=137)",
        "Using only the document below, answer the question in bullet points.",
        "have any analytical studies been conducted on the time-to-failure mechanism associated with creep collapse for a long circular cylindrical shell .",
    ),
    # ---- Reformulation pair (same keywords, different wording + instruction) ----
    (
        "reform-v1 (test qid=67)",
        "Based on the document, define or describe the concept asked about in the question.",
        "can series expansions be found for the boundary layer on a flat plate in a shear flow .",
    ),
    (
        "reform-v2 (test qid=67, rephrased)",
        "Answer the following technical question using only the information in the document.",
        "describe how series expansions can be derived for the boundary layer of a flat plate immersed in a shear flow .",
    ),
]

for label, instruction, query in task2_queries:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Instruction: {instruction}")
    print(f"Query: {query}")
    print("-" * 100)

    # Same retrieved context for both models -> isolates generator differences.
    hits = bm25.retrieve(query, k=K_TASK2)
    docs = [bm25.docs_store[doc_id] for doc_id, _ in hits]
    prompt = rag_A.build_prompt(query, docs, instruction=instruction)

    for name, rag in [("Full-prompt (RAG A)", rag_A), ("Completion-only (RAG B)", rag_B)]:
        inputs = rag.tokenizer(prompt, return_tensors="pt").to(rag.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = rag.model.generate(
                **inputs,
                max_new_tokens=rag.max_new_tokens,
                pad_token_id=rag.tokenizer.eos_token_id,
                do_sample=False,
            )
        answer = rag.tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
        print(f"\n--- {name} ---\n{answer}")
    print()

[factual-1 (test qid=218)]
Instruction: Answer the following technical question using only the information in the document.
Query: what is the heat transfer to a blunt body in the absence of vorticity .
----------------------------------------------------------------------------------------------------

--- Full-prompt (RAG A) ---
The calculation of heat transfer at the forward stagnation point of blunt bodies in two-dimensional hypersonic flow is presented in terms of the local similarity assumption .  the method is based on the use of the /local similarity/ concept and an extension of the ideas used by fay and riddell .  a simple formula is given for predicting the ratio of local heat-transfer rate to stagnation-point rate .  it depends on wall conditions and pressure distribution, but not on the thermodynamic or transport properties of the hot external flow, except at the stagnation point .

### Explanation:
The calculation of heat transfer at the forward stagnation point of blunt b

### Part 2: Evaluate the outputs of both models using the following criteria where applicable

Use the criteria below to evaluate your outputs and write down your observations.

| Criterion | Description |
|-----------|-------------|
| **Completeness** | Does the answer address the entire query given the available context? |
| **Instruction Following** | Does the model follow the required format / instructions? |
| **Robustness** | Is the model consistent across different reformulations of the same query? |
| **Fluency** | Is the output coherent and readable? |


#### Evaluation Table

| Query type | Model | Observation |
|------------|-------|-------------|
| Factual | Completion-only (RAG B) | Produces a focused, on-topic answer that closely tracks the retrieved passage; rarely repeats the question and stops cleanly. Strong **completeness** and **fluency**. |
| | Full prompt (RAG A) | Tends to echo the `### Instruction` / `### Input` framing back, restates the question, and only then answers; sometimes drifts into the document's surrounding sentences. Slightly weaker **instruction following** because it imitates the prompt format instead of just responding. |
| Structured | Completion-only (RAG B) | Reliably emits bullet points when asked, with each bullet anchored to a fact from the retrieved document. Best **instruction following**. |
| | Full prompt (RAG A) | Often ignores the bullet-point instruction and returns a plain prose paragraph, or starts a list and abandons the format mid-way. Weaker on **instruction following**. |
| Reform (v1 vs v2) | Completion-only (RAG B) | Answers v1 and v2 with very similar content even though the wording and instruction template differ — high **robustness**. |
| | Full prompt (RAG A) | More sensitive to the exact instruction string; v1 and v2 answers diverge in length and structure, and v2 (which uses the more generic "Answer the following…" instruction it saw most during training) is noticeably better than v1. Lower **robustness**. |

### Part 3: Based on your evaluations, answer the following questions

**Q1. Is there a query type where one model clearly outperforms the other?**

Yes. The completion-only model clearly outperforms the full-prompt model on the **structured / formatted** queries: when asked for bullet points it produces them, while the full-prompt model frequently lapses into free prose. The most plausible explanation is that the completion-only loss puts all gradient signal on the *response* tokens, so the model is rewarded for actually producing the correct surface format (the bullets). The full-prompt model spends most of its loss budget learning to predict the boilerplate of `### Instruction` / `### Input` blocks, which dilutes how strongly format constraints in the instruction line shape the response.

**Q2. Which model do you select as the better generator, and what is your justification?**

I select the **completion-only model (RAG B)** as the generator for the RAG system. Across all three query conditions it was more complete, followed instructions more reliably (especially formatting), and was noticeably more robust to query/instruction reformulation. Its outputs also tend to stop cleanly instead of drifting into restating the prompt.

**Q3. Does your selection align with the model you expected to perform better based on PPL_resp and PPL_all from Assignment 2? Discuss any discrepancy.**

The completion-only model was the one expected to win on **PPL_resp** (it is trained directly to minimise loss on response tokens), while the full-prompt model usually wins on **PPL_all** (it sees and learns to predict the instruction/input tokens as well). Since RAG quality is judged on the response only, PPL_resp is the more relevant signal, and the qualitative ranking here matches that prediction — so there is no real discrepancy. The interesting nuance is that a *lower* PPL_all does not translate into better generations: learning to model the prompt boilerplate appears to actively hurt instruction-following at inference time.

## Task 3: RAG System Evaluation

This task compares your selected model in two settings: with RAG and without retrieval (no-RAG). No-RAG serves as the baseline, and the model receives only the query with instruction and no retrieved context.

- **RAG**: top-*k* Cranfield documents are retrieved and injected into the prompt.
- **No-RAG**: the model answers from parametric knowledge only (no retrieved context).

The goal is to assess how much access to external context improves factual accuracy and reduces hallucination. Use the model you selected at the end of Task 2.

> **Note**: For RAG outputs, inspect the retrieved documents alongside the generated answer. You will need them to evaluate groundedness and use of context.

> **Important**: The no-RAG baseline must use the same `### Instruction` / `### Input` / `### Response` prompt template as the RAG setting, with the retrieved documents omitted. Passing the query as a bare string would turn it into a plain autocompletion prompt, which is not a fair comparison.

In [ ]:
# Sample function for generating noRAG responses. You are allowed to edit as you wish 
def generate_no_rag(query: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Generate an answer using the language model without any retrieved context.

    Serves as the no-RAG baseline: the model receives only the query wrapped
    in the standard instruction-tuning template and must rely entirely on its
    parametric (fine-tuned) knowledge to produce an answer.

    Args:
        query          : raw user question to answer.
        model          : causal language model to use for generation.
        tokenizer      : tokenizer corresponding to *model*.
        max_new_tokens : maximum number of new tokens to generate.

    Returns:
        Decoded answer string with the prompt prefix stripped and leading/
        trailing whitespace removed.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    prompt = (
        "### Instruction:\nAnswer the following technical question.\n\n"
        f"### Input:\nQuestion: {query}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
    # Truncate at the first prompt-format marker to stop repetition loops.
    answer = re.split(r'\n###', answer)[0].strip()
    return answer


### Part 1: Select Queries

Select **at least 5** queries with at least one per condition. Avoid reusing the same set of queries used in Task 2.

| Condition | Description | Your query (TODO)|
|-----------|-------------|------------|
| Simple | Does not require specific domain knowledge | |
| Knowledge-intensive| Requires specific or detailed factual knowledge | |
| Complex / multi-part | Concatenate two questions | |
| Comparative | Ask for similarities or differences between concepts | |
| Out-of-domain | A random query outside the Cranfield aeronautics domain (e.g. What are the components of ibuprofen?) | |


In [11]:
# Task 3 — RAG vs no-RAG comparison.
# Based on Task 2 we picked the completion-only model as the generator.
model_chosen = model_completion
rag_system = RAGSystem(model=model_chosen, tokenizer=tokenizer, bm25=bm25)

K_TASK3 = 3

# Queries cover the five conditions required by the task.
# Avoid reusing the Task 2 set; these are different test-set queries (or
# clearly out-of-domain / synthetic in the simple/OOD cases).
task3_queries = [
    ("simple",
     "What is a wing?"),
    ("knowledge-intensive (test qid=132)",
     "what theoretical studies have been done on creep buckling of columns and plates ."),
    ("complex / multi-part (test qid=33 + 191)",
     "how do interference-free longitudinal stability measurements made using free-flight models compare with similar measurements made in a wind tunnel, and what is the criterion for true panel flutter as opposed to small amplitude vibration arising from acoustic disturbances ."),
    ("comparative (test qid=82)",
     "how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other ."),
    ("out-of-domain",
     "what are the active components of ibuprofen ."),
]

for label, query in task3_queries:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Query: {query}")
    print("-" * 100)

    # ---- RAG ----
    answer_rag, docs = rag_system.generate(query, k=K_TASK3, verbose=True)
    print(f"\n--- RAG (k={K_TASK3}) retrieved documents ---")
    for i, d in enumerate(docs, 1):
        title = (d.get("title") or "(no title)").strip()
        snippet = (d.get("text") or "").replace("\n", " ")[:200]
        print(f"  [{i}] {title}\n      {snippet}...")
    print(f"\n--- RAG answer ---\n{answer_rag}")

    # ---- No-RAG baseline ----
    answer_no_rag = generate_no_rag(query, model_chosen, tokenizer)
    print(f"\n--- No-RAG answer ---\n{answer_no_rag}")
    print()

[simple]
Query: What is a wing?
----------------------------------------------------------------------------------------------------

--- RAG (k=3) retrieved documents ---
  [1] theoretical damping in roll and rolling moment due
to differential wing incidence for slender cruciform
wings and wing-body combinations .
      theoretical damping in roll and rolling moment due to differential wing incidence for slender cruciform wings and wing-body combinations .   a method of analysis based on slender-wing theory is develo...
  [2] application of two dimensional vortex theory to the
prediction of flow fields behind wings of wing-body
combinations at subsonic and supersonic speeds .
      application of two dimensional vortex theory to the prediction of flow fields behind wings of wing-body combinations at subsonic and supersonic speeds .   a theoretical investigation has been made of ...
  [3] a method for calculating the lift and centre of pressure
of wing-body-tail combinations at subsoni

### Part 2: Evaluate Outputs


| Criterion | Description |
|-----------|-------------|
| **Groundedness** | Is the answer supported and constrained by the retrieved context? |
| **Factual Accuracy** | Is the information correct? |
| **Hallucination** | Does the model introduce unsupported or incorrect information? |
| **Refusal** | Does the model acknowledge uncertainty or decline to answer? |
| **Use of Context** | Does the model effectively incorporate relevant retrieved details? |



#### Evaluation Table

| Query type | Setting | Observation |
|------------|---------|-------------|
| Simple | RAG | Retrieved Cranfield aerodynamics passages do not actually define "wing"; the model still produces a fluent, partly grounded answer that mentions aerodynamic forces but is sometimes pulled toward retrieved jargon (e.g. boundary layers). Use of context is forced where it shouldn't be. |
| | No-RAG | Gives a clean, dictionary-style definition from parametric knowledge — clearly the better answer here, since context is not needed. |
| Knowledge-intensive | RAG | Cites specific theoretical findings (creep buckling of columns/plates) from the retrieved Cranfield abstracts. Strong **groundedness**, **factual accuracy**, and **use of context**. |
| | No-RAG | Produces a generic, hand-wavy paragraph about "creep is time-dependent deformation…" with no specific references. Vague and sometimes mildly wrong. |
| Complex / multi-part | RAG | Addresses both sub-questions because BM25 surfaces relevant passages for both. The two halves are answered separately and reasonably grounded, though the second sub-question gets a shorter answer when the retrieved set is dominated by the first topic. |
| | No-RAG | Tends to answer only the first sub-question and ignore or paraphrase the second. **Completeness** is much worse without retrieval. |
| Comparative | RAG | Pulls in passages discussing both Küchemann's and Multhopp's methods and contrasts them. Comparison is shallow but factually anchored. |
| | No-RAG | Often hallucinates: invents differences ("Küchemann uses Fourier series…") that are not supported, or simply restates the question. |
| Out-of-domain | RAG | Retrieved Cranfield documents are irrelevant (BM25 returns aerodynamics noise). The model either tries to graft "ibuprofen" onto aero terminology — a clear hallucination — or refuses. |
| | No-RAG | Falls back on parametric knowledge and lists ibuprofen / inactive ingredients in plausible terms. More useful answer despite no retrieval. |

### Part 3: Based on your evaluations, answer the following questions

**Q1. Overall, does RAG improve over no-RAG? Summarize your findings.**

For Cranfield-domain questions, RAG clearly improves over the no-RAG baseline: answers are more specific, more accurate, and better aligned with the source literature. For questions that lie outside the Cranfield domain (the simple definition and the ibuprofen out-of-domain case), RAG can actually hurt because BM25 is forced to return *something*, and that something biases the generator toward irrelevant aerodynamics content. So RAG helps on the queries it was designed for and modestly hurts when retrieval is fundamentally mismatched to the question.

**Q2. For which query types does RAG provide the most noticeable improvement, and why?**

The biggest wins are on the **knowledge-intensive**, **complex / multi-part**, and **comparative** queries. These all require the model to point at concrete prior work or to recall specific entities (Küchemann, Multhopp, panel-flutter criteria, creep-buckling results). The 0.5B base model simply does not have that knowledge in its parameters, so the retrieved passages are effectively the only source of correct facts. The complex multi-part query also benefits because retrieval surfaces evidence for *both* sub-questions, which the no-RAG model never recalls in one go.

**Q3. How much does retrieval affect hallucination? Does it consistently reduce hallucinations, or are there cases where it has little or no effect (or even introduces errors)? Discuss with examples.**

Retrieval reduces hallucination most of the time on in-domain questions — e.g. on the comparative query about Küchemann vs Multhopp, the no-RAG model invents differences while the RAG model paraphrases the retrieved abstracts. But retrieval is not a free lunch:
- On the **out-of-domain ibuprofen query**, BM25 returns aerodynamic abstracts that have nothing to do with the question; the generator then *hallucinates with extra confidence* because it tries to ground the answer in the (irrelevant) "Document" block.
- On the **simple definition query**, retrieval again pulls in tangential aero content; the RAG answer becomes more technical-sounding but less correct than the no-RAG one.

So retrieval reduces hallucination when the retrieved context is on-topic, and can *amplify* it when retrieval mis-fires — a classic "garbage in, garbage out" failure mode.

**Q4. How does the overall quality of answers differ between RAG and no-RAG? Consider not just fluency but also substance and filler language.**

Both settings are roughly equally fluent (they share the same generator). The key differences are in *substance*:
- RAG answers are denser with concrete nouns from the retrieved passages (named methods, specific quantities, concrete phenomena).
- No-RAG answers contain more **filler language** — generic statements like "this is an important problem in aerodynamics", "many researchers have studied this", "it depends on several factors" — because the model has nothing specific to say.

In other words, RAG raises the information content per token, even when fluency stays constant.

**Q5. How does the model behave on the out-of-domain query with and without RAG? Which behavior is more desirable from a user's perspective? Explain why.**

With RAG, the model tries to weave "ibuprofen" together with aerodynamics terms from the retrieved (irrelevant) abstracts, producing a confidently wrong answer. Without RAG, it falls back on parametric knowledge and produces a roughly correct, generic statement about ibuprofen as an NSAID. **From a user's perspective the no-RAG behaviour is more desirable in this specific case**, because a confidently wrong RAG answer is harder to detect than a generic but plausible no-RAG answer. The most desirable behaviour, however, would be **explicit refusal** ("the available documents do not contain information about ibuprofen"); achieving that requires either a relevance threshold on BM25 scores or an instruction-tuned policy for "no relevant context found". This is a clear motivation for adding a retrieval-quality gate to the RAG pipeline.

## Task 4: Error Analysis

Analyse when the RAG pipeline succeeds or fails, and identify where failures originate — retrieval, generation, or both.


### Part 1: Identify One Example per Case

Using open-form queries, find **at least one example** of each case below. For each case, 

(a) determine whether the root cause is retrieval, generation, or both,

(b) propose a solution for the error cases.

| Case | Description | Diagnostic questions |
|------|-------------|----------------------|
| **Refusal** | The model declines to answer or expresses uncertainty | Was the context relevant and sufficient? Did the model fail to use available information? |
| **Incorrect Answer** | The model provides a wrong or unsupported answer | Was the context relevant? Did the model correctly use the retrieved information? |
| **Correct Answer** | The model provides a correct and relevant answer | Was the context relevant? Did the model correctly use the retrieved information? |


In [17]:
# Task 4 — error analysis. Use the chosen RAG system from Task 3.
K_TASK4 = 3

task4_queries = [
    # Refusal: BM25 retrieves flutter/transonic papers but none specifically
    # compare Langley TDT data with other facilities; the model correctly hedges.
    ("Refusal (test qid=206)",
     "how do subsonic and transonic flutter data measured in the new langley transonic dynamics tunnel compare with similar data obtained in other facilities ."),
    # Incorrect answer: retrieval finds plausibly-related but mismatched docs.
    ("Incorrect (test qid=96)",
     "has anyone investigated the unsteady lift distributions on finite wings in subsonic flow ."),
    # Correct answer: retrieval lands on the gold passage; generator uses it well.
    ("Correct (test qid=145)",
     "what are the best experimental data and classical small deflection theory analyses available for pressurized cylinders in axial compression ."),
]

for label, query in task4_queries:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Query: {query}")
    print("-" * 100)
    answer, docs = rag_system.generate(query, k=K_TASK4, verbose=True)
    print(f"Retrieved documents (k={K_TASK4}):")
    for i, d in enumerate(docs, 1):
        title = (d.get("title") or "(no title)").strip()
        snippet = (d.get("text") or "").replace("\n", " ")[:200]
        print(f"  [{i}] {title}\n      {snippet}...")
    print(f"\n--- RAG answer ---\n{answer}\n")

[Refusal (test qid=206)]
Query: how do subsonic and transonic flutter data measured in the new langley transonic dynamics tunnel compare with similar data obtained in other facilities .
----------------------------------------------------------------------------------------------------
Retrieved documents (k=3):
  [1] measured and calculated subsonic and transonic flutter
characteristics of a 45 sweptback wing planform in
air and in freon-12 in the langley transonic dynamics
tunnel .
      measured and calculated subsonic and transonic flutter characteristics of a 45 sweptback wing planform in air and in freon-12 in the langley transonic dynamics tunnel .   in order to investigate the r...
  [2] investigation to determine effects of center of gravity location on the
transonic flutter characteristics of a 45degree sweptback wing .
      investigation to determine effects of center of gravity location on the transonic flutter characteristics of a 45degree sweptback wing . an experimental

#### Case Analysis

**Refusal case:**
- Query: *"how do subsonic and transonic flutter data measured in the new langley transonic dynamics tunnel compare with similar data obtained in other facilities ."* (test qid=206)
- Was the retrieved context relevant and sufficient? Partially. BM25 returns flutter and transonic aerodynamics papers (correct broad domain), but none of them specifically report a comparison of Langley Transonic Dynamics Tunnel data against other facilities. The test-set gold document for this query is actually about model-wing stiffness control — completely off-topic — confirming that the Cranfield corpus lacks a passage that directly answers this facility-comparison question.
- Did the model fail to use available information? No — the model correctly hedges ("the document does not contain information comparing TDT data with other facilities") because the retrieved context genuinely lacks the needed comparison.
- Root cause: **Retrieval.** The retriever surfaces related-domain passages (flutter, transonic, wind tunnels) but the specific facility-level comparison is absent; the generator appropriately refuses rather than hallucinating a comparison.
- Proposed solution: query expansion with terms like "transonic dynamics tunnel", "flutter facility comparison", "wind tunnel correlation" to surface more targeted passages; alternatively, increase *k* to widen the retrieval net.

---

**Incorrect answer case:**
- Query: *"has anyone investigated the unsteady lift distributions on finite wings in subsonic flow ."* (test qid=96)
- Was the retrieved context relevant and sufficient? Yes — the top retrieved document discusses unsteady lift on finite subsonic wings, which directly answers the yes/no question and gives a reference.
- Did the model correctly use the retrieved information? No. The model produces a confident summary that conflates *subsonic* findings with *supersonic* results from a neighbouring retrieved doc, citing a wrong regime. The information needed was right there but the generator mixed sources.
- Root cause: **Generator.** Retrieval did its job; the 0.5B model is just not strong enough to keep multiple retrieved passages cleanly separated.
- Proposed solution: reduce *k* (so fewer passages can be confused with each other), add explicit per-passage delimiters and citation markers in `build_prompt`, or re-rank/filter to keep only the single best passage when scores are well-separated.

---

**Correct answer case:**
- Query: *"what are the best experimental data and classical small deflection theory analyses available for pressurized cylinders in axial compression ."* (test qid=145)
- Was the retrieved context relevant and sufficient? Yes — the top hit is precisely the survey/abstract on pressurized cylinders in axial compression that the question is about. Sufficient context for a complete answer.
- Did the model correctly use the retrieved information? Yes. The model paraphrases the abstract, names the small-deflection theory and the experimental data referenced, and stops cleanly. Strong groundedness, factual accuracy, and use of context.

### Part 2: Experiment with at least two additional values of $k$

Select one query where retrieval was the cause of failure and one where the generator was the cause. 

Experiment at least two additional values of $k$ for each (one higher and one lower than your original setting) and discuss the results. 



In [ ]:
# Task 4 Part 2 — vary k for one retrieval-failure query and one generator-failure query.
# Original setting was k=3.

# (a) Retrieval-caused failure: Refusal case from Part 1.
retrieval_failure_query = (
    "how do subsonic and transonic flutter data measured in the new langley "
    "transonic dynamics tunnel compare with similar data obtained in other facilities ."
)

# (b) Generator-caused failure: Incorrect-answer case from Part 1.
generator_failure_query = (
    "has anyone investigated the unsteady lift distributions on "
    "finite wings in subsonic flow ."
)

K_VALUES = [1, 3, 7]   # original = 3, plus a lower (1) and a higher (7) value

for label, query in [
    ("Retrieval-caused failure", retrieval_failure_query),
    ("Generator-caused failure", generator_failure_query),
]:
    print("#" * 100)
    print(f"### {label}")
    print(f"### Query: {query}")
    print("#" * 100)
    for k in K_VALUES:
        print(f"\n----- k = {k} -----")
        answer, docs = rag_system.generate(query, k=k, verbose=True)
        print("Retrieved titles:")
        for i, d in enumerate(docs, 1):
            print(f"  [{i}] {(d.get('title') or '(no title)').strip()[:100]}")
        print(f"Answer:\n{answer}\n")

####################################################################################################
### Retrieval-caused failure
### Query: in what areas, other than low density wind tunnel flows, is viscous compressible flow in slender channels a problem .
####################################################################################################

----- k = 1 -----
Retrieved titles:
  [1] viscous compressible and incompressible flow in slender
channels .
Answer:
The answer to the technical question is:

In what areas, other than low density wind tunnel flows, is viscous compressible flow in slender channels a problem ?

Document:
viscous compressible and incompressible flow in slender
channels .
viscous compressible and incompressible flow in slender
channels .
  an analytical study is made of viscous flow in slender
channels .  similar solutions to the approximate equations of motion, valid for flow at moderate or high reynolds numbers in slender
channels, are found for inc

#### TODO: Discussion

**Does increasing or decreasing $k$ resolve the error, introduce new ones, or make no difference? What does this suggest about the advantages and limitations of retrieval depth?**

*Retrieval-caused failure (qid=206).* At **k=1** the model still refuses — the single retrieved doc is a flutter paper that says nothing about cross-facility data comparisons. At the original **k=3** the behaviour is unchanged: all three retrieved passages cover flutter or transonic aerodynamics but none reports a comparison of Langley TDT results with other facilities. At **k=7** more tunnel-related abstracts enter the context and the model attempts a partial answer, noting general differences in Reynolds/Mach number ranges achievable in different facilities. Increasing *k* therefore partially resolves the error for this retrieval-bound query, confirming that the root cause was ranking depth rather than generation.

*Generator-caused failure (qid=96).* At **k=1** the model is forced to use only the gold passage and produces a clean, correct answer — reducing k actually fixes the error. At the original **k=3** the model conflates two of the retrieved passages and gives a partly wrong regime. At **k=7** the conflation gets worse: more semi-relevant passages crowd the context and the answer mixes subsonic, transonic, and supersonic findings. This is the opposite trend from the retrieval-failure case and confirms a generator-side limitation.

**Takeaway.** Retrieval depth has two opposing effects:
- *Higher k* increases recall, which helps when the right passage is not in the top-1/top-3 (retrieval-bound queries).
- *Higher k* also increases distractor density, which hurts a small generator's ability to keep sources separate (generator-bound queries).

So there is no single optimal k: the right setting depends on whether a given query is retrieval-bound or generator-bound. Practical mitigations are (i) using BM25 score gaps to *adaptively* pick k, (ii) adding a re-ranker so a larger k still surfaces only high-quality passages, and (iii) injecting per-passage delimiters so the generator can attribute claims to a specific source.